# Data Pipeline with Zero-Demand Weeks

**Goal:** Address selection bias by including weeks with zero sales.

**Problem:**
- Current panel only has weeks with sales
- If high price → no sales → week is missing from data
- This biases elasticity estimates toward zero

**Solution:**
1. Create full product × week grid
2. Impute prices using forward fill
3. Set demand = 0 for missing weeks
4. Recalculate all features

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Paths
DATA_DIR = Path('../data/processed')
OUT_DIR = Path('../data/processed')

print("Setup complete")

Setup complete


../.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Original Panel

In [2]:
# Load original panel
panel_orig = pd.read_csv(DATA_DIR / 'panel.csv')
panel_orig['week'] = pd.to_datetime(panel_orig['week'])

print(f"Original panel shape: {panel_orig.shape}")
print(f"Products: {panel_orig['product_id'].nunique()}")
print(f"Weeks: {panel_orig['week'].nunique()}")
print(f"Date range: {panel_orig['week'].min()} to {panel_orig['week'].max()}")
print(f"\nSplit distribution:")
print(panel_orig['split'].value_counts())

Original panel shape: (17970, 43)
Products: 1218
Weeks: 89
Date range: 2016-10-03 00:00:00 to 2018-08-27 00:00:00

Split distribution:
split
train    11708
val       3392
test      2870
Name: count, dtype: int64


In [3]:
# Check current gaps in data
gap_stats = panel_orig.groupby('product_id').agg(
    n_weeks=('week', 'nunique'),
    min_week=('week', 'min'),
    max_week=('week', 'max')
)
gap_stats['potential_weeks'] = ((gap_stats['max_week'] - gap_stats['min_week']).dt.days / 7 + 1).astype(int)
gap_stats['missing_weeks'] = gap_stats['potential_weeks'] - gap_stats['n_weeks']
gap_stats['fill_rate'] = gap_stats['n_weeks'] / gap_stats['potential_weeks']

print("Gap statistics per product:")
print(f"  Mean fill rate: {gap_stats['fill_rate'].mean():.1%}")
print(f"  Median fill rate: {gap_stats['fill_rate'].median():.1%}")
print(f"  Total missing weeks: {gap_stats['missing_weeks'].sum():,}")
print(f"  Total observed weeks: {gap_stats['n_weeks'].sum():,}")
print(f"  Expected total with zeros: {gap_stats['potential_weeks'].sum():,}")

Gap statistics per product:
  Mean fill rate: 35.7%
  Median fill rate: 30.9%
  Total missing weeks: 37,036
  Total observed weeks: 17,970
  Expected total with zeros: 55,006


## 2. Create Full Product × Week Grid

In [4]:
def create_full_grid(panel):
    """Create full product × week grid from first to last sale for each product."""
    
    # Get all unique weeks from original data (preserves original weekday)
    all_weeks = panel['week'].sort_values().unique()
    
    # Build full grid
    full_rows = []
    
    for product_id in tqdm(panel['product_id'].unique(), desc='Creating grid'):
        prod_data = panel[panel['product_id'] == product_id]
        prod_min = prod_data['week'].min()
        prod_max = prod_data['week'].max()
        
        # Get weeks for this product (from first to last sale)
        prod_weeks = all_weeks[(all_weeks >= prod_min) & (all_weeks <= prod_max)]
        
        for week in prod_weeks:
            full_rows.append({
                'product_id': product_id,
                'week': week
            })
    
    return pd.DataFrame(full_rows)

# Normalize weeks to start of day for proper merging
panel_orig['week'] = pd.to_datetime(panel_orig['week']).dt.normalize()

# Check weekday of original data
first_week = panel_orig['week'].min()
print(f"Original data first week: {first_week} ({first_week.day_name()})")

# Create full grid using original weeks
full_grid = create_full_grid(panel_orig)

# Verify week formats match
print(f"\nPanel orig week sample: {panel_orig['week'].iloc[0]} (type: {type(panel_orig['week'].iloc[0])})")
print(f"Full grid week sample: {full_grid['week'].iloc[0]} (type: {type(full_grid['week'].iloc[0])})")

# Check if weeks match
orig_weeks_set = set(panel_orig['week'].unique())
grid_weeks_set = set(full_grid['week'].unique())
print(f"\nWeek overlap: {len(orig_weeks_set & grid_weeks_set)} / {len(orig_weeks_set)} original weeks")

print(f"\nFull grid shape: {full_grid.shape}")
print(f"Original rows: {len(panel_orig):,}")
print(f"New rows to add: {len(full_grid) - len(panel_orig):,}")
print(f"Expansion factor: {len(full_grid) / len(panel_orig):.2f}x")

Original data first week: 2016-10-03 00:00:00 (Monday)


Creating grid:   0%|          | 0/1218 [00:00<?, ?it/s]

Creating grid:  11%|█         | 134/1218 [00:00<00:00, 1339.62it/s]

Creating grid:  22%|██▏       | 268/1218 [00:00<00:00, 1300.09it/s]

Creating grid:  33%|███▎      | 399/1218 [00:00<00:00, 1303.31it/s]

...

Creating grid:  65%|██████▌   | 797/1218 [00:00<00:00, 1278.66it/s]

Creating grid:  77%|███████▋  | 940/1218 [00:00<00:00, 1324.37it/s]

Creating grid:  89%|████████▉ | 1085/1218 [00:00<00:00, 1361.12it/s]

Creating grid: 100%|██████████| 1218/1218 [00:00<00:00, 1314.08it/s]


Panel orig week sample: 2017-02-27 00:00:00 (type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>)
Full grid week sample: 2017-02-27 00:00:00 (type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>)

Week overlap: 89 / 89 original weeks

Full grid shape: (54742, 2)
Original rows: 17,970
New rows to add: 36,772
Expansion factor: 3.05x


## 3. Merge and Fill Missing Values

In [5]:
# Merge full grid with original data
panel_full = full_grid.merge(
    panel_orig,
    on=['product_id', 'week'],
    how='left'
)

# Mark imputed rows
panel_full['is_imputed'] = panel_full['demand'].isna().astype(int)

print(f"Panel with zeros shape: {panel_full.shape}")
print(f"Imputed rows: {panel_full['is_imputed'].sum():,} ({panel_full['is_imputed'].mean():.1%})")
print(f"Observed rows: {(1 - panel_full['is_imputed']).sum():,}")

Panel with zeros shape: (54742, 44)
Imputed rows: 36,772 (67.2%)
Observed rows: 17,970


In [6]:
# Fill demand = 0 for missing weeks
panel_full['demand'] = panel_full['demand'].fillna(0)
panel_full['n_orders'] = panel_full['n_orders'].fillna(0)

# Fill static product attributes (same for all weeks of same product)
static_cols = [
    'product_category_name', 'product_category_name_english',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'product_photos_qty', 'product_name_length', 'product_description_length',
    'P0'  # baseline price
]

for col in static_cols:
    if col in panel_full.columns:
        panel_full[col] = panel_full.groupby('product_id')[col].transform(
            lambda x: x.ffill().bfill()
        )

print("Filled demand and static columns")

Filled demand and static columns


In [7]:
# Impute prices using forward fill (then backward fill for first weeks)
price_cols = ['avg_price', 'min_price', 'max_price', 'price_std', 'price_range', 'n_unique_prices', 'freight_mean']

panel_full = panel_full.sort_values(['product_id', 'week'])

for col in price_cols:
    if col in panel_full.columns:
        # Forward fill, then backward fill for remaining NaNs
        panel_full[col] = panel_full.groupby('product_id')[col].transform(
            lambda x: x.ffill().bfill()
        )

# For imputed rows, set price_std and price_range to 0 (no variation)
panel_full.loc[panel_full['is_imputed'] == 1, 'price_std'] = 0
panel_full.loc[panel_full['is_imputed'] == 1, 'price_range'] = 0
panel_full.loc[panel_full['is_imputed'] == 1, 'n_unique_prices'] = 1

print("Imputed prices with forward fill")

Imputed prices with forward fill


## 4. Recalculate Features

In [8]:
# Sort by product and week
panel_full = panel_full.sort_values(['product_id', 'week']).reset_index(drop=True)

# Time features
panel_full['year'] = panel_full['week'].dt.year
panel_full['month'] = panel_full['week'].dt.month
panel_full['weekofyear'] = panel_full['week'].dt.isocalendar().week.astype(int)
panel_full['week_sin'] = np.sin(2 * np.pi * panel_full['weekofyear'] / 52.0)
panel_full['week_cos'] = np.cos(2 * np.pi * panel_full['weekofyear'] / 52.0)

print("Calculated time features")

Calculated time features


In [9]:
# Lag features (recalculate on full grid)
ROLLING_WINDOW = 4

# Demand lags
panel_full['demand_lag_1'] = panel_full.groupby('product_id')['demand'].shift(1)
panel_full['demand_lag_2'] = panel_full.groupby('product_id')['demand'].shift(2)
panel_full['demand_roll_4'] = panel_full.groupby('product_id')['demand'].transform(
    lambda x: x.shift(1).rolling(window=ROLLING_WINDOW, min_periods=1).mean()
)

# Price lags
panel_full['price_lag_1'] = panel_full.groupby('product_id')['avg_price'].shift(1)
panel_full['price_roll_4'] = panel_full.groupby('product_id')['avg_price'].transform(
    lambda x: x.shift(1).rolling(window=ROLLING_WINDOW, min_periods=1).mean()
)

# Gap feature (weeks since last sale > 0)
def calc_weeks_since_sale(group):
    result = []
    last_sale_week = None
    for idx, row in group.iterrows():
        if last_sale_week is None:
            result.append(0)
        else:
            result.append((row['week'] - last_sale_week).days / 7)
        if row['demand'] > 0:
            last_sale_week = row['week']
    return pd.Series(result, index=group.index)

panel_full['weeks_since_last_sale'] = panel_full.groupby('product_id', group_keys=False).apply(calc_weeks_since_sale)

print("Calculated lag features")

Calculated lag features


In [10]:
# Review features (forward fill from original data)
review_cols = ['sku_review_count', 'sku_review_mean', 'sku_share_low']

for col in review_cols:
    if col in panel_full.columns:
        panel_full[col] = panel_full.groupby('product_id')[col].transform(
            lambda x: x.ffill().fillna(0)
        )

print("Filled review features")

Filled review features


In [11]:
# Calculate r (relative price) and y (log demand)
panel_full['r'] = panel_full['avg_price'] / panel_full['P0']

# Clip r to original train quantiles
R_CLIP_LOW = 0.785  # from original pipeline
R_CLIP_HIGH = 1.283
panel_full['r_clipped'] = panel_full['r'].clip(lower=R_CLIP_LOW, upper=R_CLIP_HIGH)

# Target
panel_full['y'] = np.log1p(panel_full['demand'])

print("Calculated r and y")
print(f"\ny distribution:")
print(f"  y=0 (zero demand): {(panel_full['y'] == 0).sum():,} ({(panel_full['y'] == 0).mean():.1%})")
print(f"  y>0 (positive demand): {(panel_full['y'] > 0).sum():,} ({(panel_full['y'] > 0).mean():.1%})")

Calculated r and y

y distribution:
  y=0 (zero demand): 36,772 (67.2%)
  y>0 (positive demand): 17,970 (32.8%)


## 5. Assign Splits

In [12]:
# Split boundaries (same as original)
TRAIN_END = pd.Timestamp('2018-02-19')
VAL_END = pd.Timestamp('2018-05-07')

panel_full['split'] = np.where(
    panel_full['week'] <= TRAIN_END,
    'train',
    np.where(panel_full['week'] <= VAL_END, 'val', 'test')
)

print("Split distribution:")
print(panel_full['split'].value_counts())

print("\nZero-demand distribution by split:")
for split in ['train', 'val', 'test']:
    split_data = panel_full[panel_full['split'] == split]
    zero_rate = (split_data['demand'] == 0).mean()
    print(f"  {split}: {zero_rate:.1%} zeros")

Split distribution:
split
train    37175
val       9677
test      7890
Name: count, dtype: int64

Zero-demand distribution by split:
  train: 68.5% zeros
  val: 64.9% zeros
  test: 63.6% zeros


In [13]:
# Recalculate train statistics for usable_for_elasticity
train_mask = panel_full['split'] == 'train'
train_stats = panel_full[train_mask].groupby('product_id').agg(
    train_weeks=('week', 'nunique'),
    train_price_min=('r', 'min'),
    train_price_max=('r', 'max'),
    train_unique_prices=('avg_price', 'nunique'),
)
train_stats['train_price_span'] = train_stats['train_price_max'] - train_stats['train_price_min']

# Merge back
panel_full = panel_full.drop(columns=['train_weeks', 'train_price_span', 'train_unique_prices'], errors='ignore')
panel_full = panel_full.merge(
    train_stats[['train_weeks', 'train_price_span', 'train_unique_prices']],
    on='product_id',
    how='left'
)

# Usable for elasticity
MIN_TRAIN_WEEKS = 5
MIN_TRAIN_PRICE_SPAN = 0.02
MIN_TRAIN_UNIQUE_PRICES = 2

panel_full['usable_for_elasticity'] = (
    (panel_full['train_weeks'] >= MIN_TRAIN_WEEKS) &
    (panel_full['train_price_span'] >= MIN_TRAIN_PRICE_SPAN) &
    (panel_full['train_unique_prices'] >= MIN_TRAIN_UNIQUE_PRICES)
).astype(int)

print(f"Usable for elasticity: {panel_full[panel_full['usable_for_elasticity']==1]['product_id'].nunique()} products")

Usable for elasticity: 882 products


## 6. Final Cleanup and Save

In [14]:
# Fill any remaining NaNs
fill_cols = [
    'demand_lag_1', 'demand_lag_2', 'demand_roll_4',
    'price_lag_1', 'price_roll_4', 'weeks_since_last_sale',
    'price_std', 'price_range'
]
for col in fill_cols:
    if col in panel_full.columns:
        panel_full[col] = panel_full[col].fillna(0)

# Check for NaNs
nan_counts = panel_full.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols) > 0:
    print("Columns with NaN:")
    print(nan_cols)
else:
    print("No NaN values remaining")

Columns with NaN:
product_category_name            308
product_category_name_english    308
product_weight_g                  23
product_length_cm                 23
product_height_cm                 23
product_width_cm                  23
product_photos_qty               308
product_name_length              308
product_description_length       308
dtype: int64


In [15]:
# Summary comparison
print("\n" + "="*60)
print("SUMMARY: Original vs With Zeros")
print("="*60)

print(f"\n{'Metric':<30} {'Original':>12} {'With Zeros':>12}")
print("-" * 56)
print(f"{'Total rows':<30} {len(panel_orig):>12,} {len(panel_full):>12,}")
print(f"{'Products':<30} {panel_orig['product_id'].nunique():>12,} {panel_full['product_id'].nunique():>12,}")
print(f"{'Zero-demand rows':<30} {'0':>12} {(panel_full['demand']==0).sum():>12,}")
print(f"{'Zero-demand %':<30} {'0%':>12} {(panel_full['demand']==0).mean()*100:>11.1f}%")
print(f"{'Expansion factor':<30} {'1.0x':>12} {len(panel_full)/len(panel_orig):>11.2f}x")


SUMMARY: Original vs With Zeros

Metric                             Original   With Zeros
--------------------------------------------------------
Total rows                           17,970       54,742
Products                              1,218        1,218
Zero-demand rows                          0       36,772
Zero-demand %                            0%        67.2%
Expansion factor                       1.0x        3.05x


In [16]:
# Save
panel_full['week'] = panel_full['week'].dt.strftime('%Y-%m-%d')
panel_full.to_csv(OUT_DIR / 'panel_with_zeros.csv', index=False)

print(f"\nSaved to {OUT_DIR / 'panel_with_zeros.csv'}")
print(f"File size: {(OUT_DIR / 'panel_with_zeros.csv').stat().st_size / 1024 / 1024:.1f} MB")


Saved to ../data/processed/panel_with_zeros.csv
File size: 17.6 MB


## 7. Quick Baseline Comparison (XGBoost)

In [17]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

# Reload with correct types
panel_full = pd.read_csv(OUT_DIR / 'panel_with_zeros.csv')
panel_full['week'] = pd.to_datetime(panel_full['week'])

# Encode category
le_cat = LabelEncoder()
panel_full['category_encoded'] = le_cat.fit_transform(
    panel_full['product_category_name_english'].fillna('unknown')
)

print("Data loaded for XGBoost")

Data loaded for XGBoost


In [18]:
# Features
feature_cols = [
    'r', 'r_clipped', 'avg_price', 'P0',
    'price_lag_1', 'price_roll_4', 'price_std', 'price_range', 'min_price', 'max_price',
    'year', 'month', 'weekofyear', 'week_sin', 'week_cos',
    'demand_lag_1', 'demand_lag_2', 'demand_roll_4', 'weeks_since_last_sale',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'product_photos_qty', 'product_name_length', 'product_description_length',
    'sku_review_count', 'sku_review_mean', 'sku_share_low',
    'category_encoded'
]

# Fill NaN
for col in feature_cols:
    if col in panel_full.columns:
        panel_full[col] = panel_full[col].fillna(0)

# Split
train_mask = panel_full['split'] == 'train'
val_mask = panel_full['split'] == 'val'
test_mask = panel_full['split'] == 'test'

X_train = panel_full.loc[train_mask, feature_cols]
y_train = panel_full.loc[train_mask, 'y']
X_val = panel_full.loc[val_mask, feature_cols]
y_val = panel_full.loc[val_mask, 'y']
X_test = panel_full.loc[test_mask, feature_cols]
y_test = panel_full.loc[test_mask, 'y']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (37175, 30), Val: (9677, 30), Test: (7890, 30)


In [19]:
# Train XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'seed': 42
}

print("Training XGBoost...")
bst = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=50
)

print(f"\nBest iteration: {bst.best_iteration}")

Training XGBoost...
[0]	train-rmse:0.49966	val-rmse:0.53941


[50]	train-rmse:0.39120	val-rmse:0.44215


[77]	train-rmse:0.38134	val-rmse:0.44450



Best iteration: 27


In [20]:
# Compute corr_mse
def compute_corr_mse(y_true, y_pred, groups):
    """Compute group-corrected MSE."""
    df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'group': groups})
    
    def group_mse(g):
        if len(g) < 2:
            return 0, 0
        y_t = g['y_true'].values
        y_p = g['y_pred'].values
        y_t_c = y_t - y_t.mean()
        y_p_c = y_p - y_p.mean()
        mse = ((y_t_c - y_p_c) ** 2).mean()
        return mse * len(g), len(g)
    
    results = df.groupby('group').apply(group_mse)
    total = sum(r[0] for r in results)
    weight = sum(r[1] for r in results)
    return total / weight if weight > 0 else 0

# Predictions
y_test_pred = bst.predict(dtest)

# Metrics
test_corr_mse = compute_corr_mse(
    y_test.values,
    y_test_pred,
    panel_full.loc[test_mask, 'product_id'].values
)

print("\n" + "="*60)
print("XGBOOST COMPARISON")
print("="*60)
print(f"\n{'Dataset':<25} {'Test corr_mse':>15}")
print("-" * 42)
print(f"{'Original (no zeros)':<25} {'0.0964':>15}")
print(f"{'With zeros':<25} {test_corr_mse:>15.4f}")
print(f"{'Change':<25} {(test_corr_mse - 0.0964) / 0.0964 * 100:>14.1f}%")


XGBOOST COMPARISON

Dataset                     Test corr_mse
------------------------------------------
Original (no zeros)                0.0964
With zeros                         0.1814
Change                              88.2%


In [21]:
# Estimate elasticity from XGBoost (coefficient of log_r feature)
# Use SHAP or feature importance as proxy

importance = bst.get_score(importance_type='gain')
print("\nTop 10 features by gain:")
for feat, score in sorted(importance.items(), key=lambda x: -x[1])[:10]:
    # Map feature names
    feat_name = feature_cols[int(feat.replace('f', ''))] if feat.startswith('f') else feat
    print(f"  {feat_name:<30} {score:>10.2f}")


Top 10 features by gain:
  price_std                           24.42
  weeks_since_last_sale               23.42
  demand_roll_4                       20.40
  demand_lag_1                         7.22
  sku_review_count                     5.80
  price_lag_1                          5.30
  sku_review_mean                      4.05
  sku_share_low                        1.80
  weekofyear                           1.76
  avg_price                            1.67


## 8. Summary

In [22]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print(f"\n### Data Expansion")
print(f"- Original rows: {len(panel_orig):,}")
print(f"- With zeros: {len(panel_full):,}")
print(f"- Zero-demand rows: {(panel_full['demand']==0).sum():,} ({(panel_full['demand']==0).mean():.1%})")

print(f"\n### XGBoost Performance")
print(f"- Original: test corr_mse = 0.0964")
print(f"- With zeros: test corr_mse = {test_corr_mse:.4f}")

print(f"\n### Files Created")
print(f"- {OUT_DIR / 'panel_with_zeros.csv'}")


FINAL SUMMARY

### Data Expansion
- Original rows: 17,970
- With zeros: 54,742
- Zero-demand rows: 36,772 (67.2%)

### XGBoost Performance
- Original: test corr_mse = 0.0964
- With zeros: test corr_mse = 0.1814

### Files Created
- ../data/processed/panel_with_zeros.csv


## 9. Analyzing the corr_mse Degradation

The corr_mse got **worse** with zeros. Let's understand why.

In [23]:
# Analyze predictions for observed vs imputed rows
test_df = panel_full[test_mask].copy()
test_df['y_pred'] = y_test_pred
test_df['residual'] = test_df['y'] - test_df['y_pred']
test_df['is_zero'] = (test_df['y'] == 0).astype(int)

print("="*60)
print("PREDICTION ANALYSIS BY ROW TYPE")
print("="*60)

print("\n### Test Set Composition")
print(f"  Observed rows (is_imputed=0): {(test_df['is_imputed']==0).sum():,}")
print(f"  Imputed rows (is_imputed=1):  {(test_df['is_imputed']==1).sum():,}")

print("\n### MSE by Row Type")
for imp in [0, 1]:
    subset = test_df[test_df['is_imputed'] == imp]
    mse = ((subset['y'] - subset['y_pred'])**2).mean()
    row_type = 'Observed' if imp == 0 else 'Imputed (zeros)'
    print(f"  {row_type}: MSE = {mse:.4f}")

print("\n### Prediction Statistics")
print(f"\n  For y=0 (zero demand):")
zero_preds = test_df[test_df['y'] == 0]['y_pred']
print(f"    Mean prediction: {zero_preds.mean():.3f}")
print(f"    Should be ~0, actual range: [{zero_preds.min():.3f}, {zero_preds.max():.3f}]")

print(f"\n  For y>0 (positive demand):")
pos_preds = test_df[test_df['y'] > 0]['y_pred']
print(f"    Mean prediction: {pos_preds.mean():.3f}")
print(f"    Mean actual: {test_df[test_df['y'] > 0]['y'].mean():.3f}")

PREDICTION ANALYSIS BY ROW TYPE

### Test Set Composition
  Observed rows (is_imputed=0): 2,870
  Imputed rows (is_imputed=1):  5,020

### MSE by Row Type
  Observed: MSE = 0.3325
  Imputed (zeros): MSE = 0.1279

### Prediction Statistics

  For y=0 (zero demand):
    Mean prediction: 0.284
    Should be ~0, actual range: [-0.032, 1.898]

  For y>0 (positive demand):
    Mean prediction: 0.600
    Mean actual: 0.979


In [24]:
# Compute corr_mse for OBSERVED rows only (fair comparison with original)
observed_test = test_df[test_df['is_imputed'] == 0]

corr_mse_observed = compute_corr_mse(
    observed_test['y'].values,
    observed_test['y_pred'].values,
    observed_test['product_id'].values
)

print("\n" + "="*60)
print("FAIR COMPARISON: corr_mse on OBSERVED rows only")
print("="*60)

print(f"\n{'Metric':<40} {'Value':>10}")
print("-" * 52)
print(f"{'Original (no zeros, test set)':<40} {'0.0964':>10}")
print(f"{'With zeros (OBSERVED test rows only)':<40} {corr_mse_observed:>10.4f}")
print(f"{'With zeros (ALL test rows)':<40} {test_corr_mse:>10.4f}")

print(f"\n### Interpretation:")
if corr_mse_observed < 0.0964:
    print(f"  ✓ Training with zeros IMPROVED observed-row predictions!")
    print(f"  ✓ Model better understands price→demand relationship")
else:
    print(f"  ✗ Training with zeros did not improve observed-row predictions")
    print(f"  ✗ Zero-inflation may be hurting the model")


FAIR COMPARISON: corr_mse on OBSERVED rows only

Metric                                        Value
----------------------------------------------------
Original (no zeros, test set)                0.0964
With zeros (OBSERVED test rows only)         0.1263
With zeros (ALL test rows)                   0.1814

### Interpretation:
  ✗ Training with zeros did not improve observed-row predictions
  ✗ Zero-inflation may be hurting the model


In [25]:
# Estimate elasticity using finite differences
# For each product, compute ∂y/∂r at r=1

def estimate_elasticity_xgb(model, X_base, r_col='r', delta=0.01):
    """Estimate elasticity at current r using finite differences."""
    X_low = X_base.copy()
    X_high = X_base.copy()
    
    X_low[r_col] = X_base[r_col] - delta
    X_high[r_col] = X_base[r_col] + delta
    
    d_low = xgb.DMatrix(X_low)
    d_high = xgb.DMatrix(X_high)
    
    y_low = model.predict(d_low)
    y_high = model.predict(d_high)
    
    # dy/dr ≈ (y_high - y_low) / (2*delta)
    dy_dr = (y_high - y_low) / (2 * delta)
    
    # Elasticity = dy/dr * r/y ≈ dy/dr (since y is log and r is ~1)
    return dy_dr

# Get elasticity for test set
test_elasticity = estimate_elasticity_xgb(bst, X_test)
test_df['elasticity'] = test_elasticity

print("\n" + "="*60)
print("ELASTICITY ESTIMATES")
print("="*60)

print(f"\n### Overall Statistics")
print(f"  Mean elasticity: {test_df['elasticity'].mean():.4f}")
print(f"  Median elasticity: {test_df['elasticity'].median():.4f}")
print(f"  Std: {test_df['elasticity'].std():.4f}")

print(f"\n### Expected vs Actual")
print(f"  Expected: negative (-1 to -3 typical)")
print(f"  Got: {test_df['elasticity'].mean():.4f}")

print(f"\n### By Row Type")
for imp in [0, 1]:
    subset = test_df[test_df['is_imputed'] == imp]
    row_type = 'Observed' if imp == 0 else 'Imputed'
    print(f"  {row_type}: mean elasticity = {subset['elasticity'].mean():.4f}")


ELASTICITY ESTIMATES

### Overall Statistics
  Mean elasticity: -0.1957
  Median elasticity: 0.0000
  Std: 0.8329

### Expected vs Actual
  Expected: negative (-1 to -3 typical)
  Got: -0.1957

### By Row Type
  Observed: mean elasticity = -0.2111
  Imputed: mean elasticity = -0.1870


In [26]:
print("\n" + "="*60)
print("CONCLUSION")
print("="*60)

print("""
### Key Findings:

1. **Data expansion worked**: 17,970 → 54,742 rows (3x)
   - 67% of rows are now zero-demand weeks
   
2. **corr_mse on ALL rows got worse**: 0.0964 → 0.1814
   - This is expected: predicting zeros is hard with forward-filled prices
   - The imputed prices are noisy approximations
   
3. **Need to check**: corr_mse on OBSERVED rows only
   - This is the fair comparison with original data
   - If this improved, the hypothesis about selection bias is confirmed
   
4. **Elasticity estimates**:
   - Check if they became more negative (as expected with zeros)
   
### Recommendations:

1. For thesis: report BOTH metrics (all rows and observed-only)
2. Consider a two-stage model:
   - Stage 1: Predict P(demand > 0) - classification
   - Stage 2: Predict demand | demand > 0 - regression
3. The selection bias fix is conceptually correct, but implementation 
   needs refinement for practical improvement
""")


CONCLUSION

### Key Findings:

1. **Data expansion worked**: 17,970 → 54,742 rows (3x)
   - 67% of rows are now zero-demand weeks
   
2. **corr_mse on ALL rows got worse**: 0.0964 → 0.1814
   - This is expected: predicting zeros is hard with forward-filled prices
   - The imputed prices are noisy approximations
   
3. **Need to check**: corr_mse on OBSERVED rows only
   - This is the fair comparison with original data
   - If this improved, the hypothesis about selection bias is confirmed
   
4. **Elasticity estimates**:
   - Check if they became more negative (as expected with zeros)
   
### Recommendations:

1. For thesis: report BOTH metrics (all rows and observed-only)
2. Consider a two-stage model:
   - Stage 1: Predict P(demand > 0) - classification
   - Stage 2: Predict demand | demand > 0 - regression
3. The selection bias fix is conceptually correct, but implementation 
   needs refinement for practical improvement



## 10. Documented Results: Zero-Imputation Experiment

### Hypothesis
Including zero-demand weeks would address selection bias where high prices → no sales → missing data, leading to:
- More negative elasticity estimates (closer to true -1 to -3 range)
- Better understanding of price→demand relationship

### Method
1. Created full product × week grid (first sale to last sale for each product)
2. Imputed prices using forward fill
3. Set demand = 0 for missing weeks
4. Retrained XGBoost on expanded dataset

### Results Summary

| Metric | Original (no zeros) | With Zeros | Change |
|--------|---------------------|------------|--------|
| Dataset rows | 17,970 | 54,742 | +206% |
| Zero-demand rows | 0 | 36,772 (67%) | - |
| Test corr_mse (all) | 0.0964 | 0.1814 | **+88% worse** |
| Test corr_mse (observed only) | 0.0964 | 0.1263 | **+31% worse** |
| Mean elasticity | ~-0.5 | -0.20 | Closer to zero |

### Conclusion: Hypothesis NOT Confirmed

**The naive zero-imputation approach failed:**

1. **Prediction quality degraded** - corr_mse increased even for observed rows
2. **Elasticity did not improve** - estimates moved toward zero, not away
3. **Root causes:**
   - Forward-fill prices are noisy approximations of true listing prices
   - 67% zeros create severe class imbalance
   - Model learns to predict zeros at expense of positive demand accuracy

### Implications for Thesis

This experiment should be documented as a **negative result** demonstrating:
- Naive selection bias correction can backfire
- Price imputation quality is critical for demand modeling
- Zero-inflated data requires specialized modeling approaches (e.g., hurdle models)

**Recommendation:** Try alternative approaches:
1. Hurdle/Two-stage model (separate classification + regression)
2. Sample weighting by inverse probability
3. Focus on products with denser sales histories